# HNL mu-pi: BNB MC + off-beam + data + HNL (cafpybara)

Same frames and variables as `hnl_mupi_selection.ipynb`, with the Run 1 data added. Cuts: the full selection,
`ana.DEFAULT_CUTS` + `ana.track_cuts(..., containment_last=True)` with **per-mass** thresholds (`thresholds_m<mass>.txt`),
also passed to the loaders, so BNB, off-beam and data are loaded once per mass. Containment is not a filter: it is the last
step and splits the selected slices into a contained and an uncontained sample. The MC/data plots are made per category
(`<level>_<var>_m<mass>_<contained|uncontained>.pdf`)
(structure follows the nueCC `signal_plots` and hnlpi0 `hnl_plotting` notebooks):

* Prediction = BNB nu + CORSIKA cosmics MC (weight 1) + off-beam+light data (weight `weights_mc` = the off-beam scale to the BNB MC POT),
  all times `POT_DATA / POT_BNB`. The off-beam scale is `(1 - f) * (on-beam gates per POT * POT_BNB) / off-beam gates`, `f = config.OFFBEAM_F`,
  as in nueCC and hnlpi0.
* Data = on-beam BNB+light (Run 1 respin, 2025-02-16..19), Poisson errors. The Data/Pred band is the weighted stack's statistical
  error (BNB MC + off-beam). No systematics yet.
* HNL = step line at the simulated |U_mu4|^2 (MeVPrtl weights) scaled to the data POT, times `ana.HNL_DISPLAY_SCALE[m]`. One figure per mass.
* Data timing: `preprocess_databnb` (per-run-period calibration, drops slices without RWM timing and bad periods), `preprocess_dataoff` (off-beam calibration).
* PDFs go to `PLOT_DIR` as `<level>_<var>_m<mass>_<category>.pdf`.

In [ ]:
import sys
from pathlib import Path

sys.path[:0] = ["/exp/sbnd/app/users/sdas1/cafpyana", "/exp/sbnd/app/users/sdas1"]

from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import cafpybara.analyses.hnl_mupi as ana
from cafpybara.analyses.hnl_mupi import config

PLOT_DIR = Path("/exp/sbnd/app/users/sdas1/hnl/plots/selection_study_cafpybara_data_run1_fullsel")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
MASSES = (300, 365)
# per-mass track-cut thresholds; the loaders count PFPs with them
THR = {m: ana.read_thresholds(ana.THRESHOLD_FILE.with_name(f"thresholds_m{m}.txt")) for m in MASSES}
# full sequence, containment last (the split separator); SEL drops it: both categories
CUTS = {m: ana.DEFAULT_CUTS + ana.track_cuts(THR[m]["open_angle_max"], containment_last=True) for m in MASSES}
SEL = {m: [c for c in CUTS[m] if c.name != "contained"] for m in MASSES}
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
c = lambda *p: tuple(list(p) + [""] * (6 - len(p)))   # 6-level column key


## 1. Load (cafpyana preselection already applied)

In [ ]:
STREAM = True   # read the frames from dCache over xrootd (no /pnfs mount, e.g. on JupyterHub)
if STREAM:
    ana.ensure_token()
    url = ana.xrootd_url
else:
    url = lambda p: p
# The PFP counts are made at load time with the thresholds, so every sample is loaded once per mass
# (re-running preprocess on a loaded frame would be skipped: collapse_pfp marks itself applied).
bnb0, data0, off0, hnl0, POT_HNL, HNL_INFO = {}, {}, {}, {}, {}, {}
for m in MASSES:
    t = THR[m]
    bnb0[m], POT_BNB, _ = ana.load_mc(url(config.BNB_FILE), preprocess_fn=partial(ana.preprocess_mcbnb, thresholds=t))
    hnl0[m], POT_HNL[m], HNL_INFO[m] = ana.load_mchnl(url(config.HNL_FILES[m]), preprocess_fn=partial(ana.preprocess_mchnl, thresholds=t))
    data0[m], POT_DATA, NGATES_ON = ana.load_data(url(config.DATA_BNB_FILE), onbeam=True,
                                                  preprocess_fn=partial(ana.preprocess_databnb, thresholds=t))
    off0[m], _, NGATES_OFF = ana.load_data(url(config.DATA_OFFBEAM_FILE), onbeam=False,
                                           preprocess_fn=partial(ana.preprocess_dataoff, thresholds=t))

W = POT_DATA / POT_BNB                                   # BNB MC -> data POT
W_OFF = (1 - config.OFFBEAM_F) * (NGATES_ON / POT_DATA * POT_BNB) / NGATES_OFF   # off-beam slice -> BNB MC POT
for m in MASSES:
    off0[m][("weights_mc", "", "", "", "", "")] = W_OFF
W_HNL = {m: POT_DATA / POT_HNL[m] for m in MASSES}        # times weights_mc
m0 = MASSES[0]
print(f"Data: {len(data0[m0])} slices, POT = {POT_DATA:.3e}, on-beam gates = {NGATES_ON:.4g}; off-beam: {len(off0[m0])} slices, gates = {NGATES_OFF:.4g}")
print(f"BNB: {len(bnb0[m0])} slices, POT = {POT_BNB:.3e}; scale to data = {W:.4f}; off-beam weight = {W_OFF:.4f} (x {W:.4f})")
for m, h in hnl0.items():
    sig = h.signal == ana.signal_dict['hnl']
    print(f"m{m}: {len(h)} slices, {int(sig.sum())} HNL-matched, |U|^2 = {HNL_INFO[m]['simU']:.1e}, "
          f"expected matched slices @ {POT_DATA:.2e} POT = {h.weights_mc[sig].to_numpy().sum() * W_HNL[m]:.3g}")
    print(f"   thresholds m{m}:", THR[m])
    print(f"   BNB slices passing the per-mass selection (both categories): {len(ana.select(bnb0[m], SEL[m]))}")
print("fixes: BNB", ana.applied_fixes(bnb0[m0]), "| HNL", {m: ana.applied_fixes(h) for m, h in hnl0.items()},
      "| data", ana.applied_fixes(data0[m0]), "| off-beam", ana.applied_fixes(off0[m0]))

## 2. Cut flow
Per mass (own thresholds). Slices per sample after each cut, scaled to the data POT; off-beam and the prediction include their scale factors.
The last row (`contained`) is the contained branch of the split; the uncontained branch is the row before it minus that one.

In [ ]:
sel = lambda df, cuts: ana.select(df, cuts=cuts, savedict=True, check_preprocessed=False)
for m in MASSES:
    st = {k: {"preselection": d[m], **sel(d[m], CUTS[m])} for k, d in (("bnb", bnb0), ("off", off0), ("data", data0))}
    rows = []
    for stage, label in {"preselection": "cafpyana preselection", **{c.name: c.label for c in CUTS[m]}}.items():
        b, o, d = len(st["bnb"][stage]) * W, len(st["off"][stage]) * W_OFF * W, len(st["data"][stage])
        rows.append({"cut": label, "BNB MC": b, "off-beam": o, "prediction": b + o, "data": d, "data/pred": d / (b + o)})
    print(f"m{m} thresholds")
    display(pd.DataFrame(rows).set_index("cut").round(3))
    display(ana.cutflow({"bnb": bnb0[m], f"m{m}": hnl0[m]}, CUTS[m], weights={"bnb": W, f"m{m}": W_HNL[m]}).round(3))
# selected samples for the plots: full selection (no containment), then split [m][category]
CATS = ("contained", "uncontained")
def by_cat(df):
    c = df["n_uncontained"].to_numpy(float).ravel() == 0
    return {"contained": df[c], "uncontained": df[~c]}
pred = {m: by_cat(pd.concat([ana.select(bnb0[m], SEL[m]), ana.select(off0[m], SEL[m])], ignore_index=True)) for m in MASSES}
data = {m: by_cat(ana.select(data0[m], SEL[m])) for m in MASSES}
hnl = {m: ana.select(h, SEL[m]) for m, h in hnl0.items()}
hnl_sig = {m: by_cat(h[h.signal == ana.signal_dict["hnl"]]) for m, h in hnl.items()}
for m in MASSES:
    print(f"m{m}:", {k: f"data {len(data[m][k])}, pred {len(pred[m][k]) and pred[m][k].weights_mc.to_numpy().sum() * W:.1f}" for k in CATS})

## 3. Plot helper

In [ ]:
MAX_BINS = 15   # the full selection keeps few data slices per category: coarse bins

def plot(name, col, xlabel, bins=None, logy=False):
    """One PDF per mass and category (contained / uncontained), each with its own selection.
    bins=None: MAX_BINS bins over the 0.5-99.5 percentiles of that mass's prediction, data and HNL (both categories,
    so the two categories share bins). Fixed bins with more than MAX_BINS bins are merged k at a time, k the smallest
    divisor of the bin count that gives <= MAX_BINS (equal widths, range unchanged)."""
    for m in MASSES:
        b = bins
        if b is None:
            b = ana.auto_bins([*pred[m].values(), *data[m].values(), *hnl_sig[m].values()], col, nb=MAX_BINS)
            if b is None:
                print("no valid entries:", name, m); continue
        elif len(b) - 1 > MAX_BINS:
            n = len(b) - 1
            k = next((d for d in range(2, n + 1) if n % d == 0 and n // d <= MAX_BINS), n)
            b = np.asarray(b)[::k]
        s = ana.HNL_DISPLAY_SCALE[m]
        for cat in CATS:
            if len(pred[m][cat]) == 0 and len(data[m][cat]) == 0:
                print("empty:", name, m, cat); continue
            try:
                res = ana.plot_mc_hnl_data(pred[m][cat], hnl_sig[m][cat], data[m][cat], col, b, scale_nu=W, scale_hnl=W_HNL[m] * s,
                                           hnl_label=ana.hnl_label(m, HNL_INFO[m]["simU"], s), log_y=logy,
                                           overflow=False, xlabel=xlabel, title=" ", counts=True,
                                           ylabel=f"Slices / bin ({POT_DATA:.2e} POT)",
                                           legend_kwargs={"fontsize": 8},
                                           savefig=PLOT_DIR / f"{name}_m{m}_{cat}.pdf")
            except Exception as e:   # a sparse category must not stop the other plots
                print("plot failed:", name, m, cat, repr(e)); plt.close("all"); continue
            plt.show()
            plt.close(res[0])

## 4. Slice-level variables (after the cuts)

In [ ]:
P = 18.936  # bunch period [ns]
SLICE_VARS = [
    ("nu_score",            c("slc", "nu_score"),                       "Pandora ν score",           np.linspace(0.5, 1, 51),  True),
    ("bcfm_score",          c("slc", "barycenterFM", "score"),          "BCFM score",                np.linspace(0, 2, 81),    True),
    ("bcfm_score_zoom",     c("slc", "barycenterFM", "score"),          "BCFM score (zoom)",         np.linspace(0, 0.3, 61),  True),
    ("bcfm_deltaZ",         c("slc", "barycenterFM", "deltaZ"),         "BCFM ΔZ [cm]",              None,                     True),
    ("bcfm_chi2",           c("slc", "barycenterFM", "chi2"),           "BCFM χ²",                   None,                     True),
    ("bcfm_chargeTotal",    c("slc", "barycenterFM", "chargeTotal"),    "BCFM total charge",         None,                     True),
    ("bcfm_flashPEs",       c("slc", "barycenterFM", "flashPEs"),       "BCFM flash PEs",            None,                     True),
    ("bcfm_flashTime_corr", c("slc", "barycenterFM", "flashTime"),      "BCFM flash time, generator-corrected [µs]", None, False),
    ("bcfm_flashTime_phase", c("slc", "barycenterFM", "flashTime_calib_mod"), "(flash time - vertex ToF) mod 18.936 ns [ns]",
     np.linspace(0, P, 25), False),
    ("opt0_score",          c("slc", "opt0", "score"),                  "opt0 score",                None,                     True),
    ("opt0_time",           c("slc", "opt0", "time"),                   "opt0 time [µs]",            None,                     False),
    ("opt0_measPE",         c("slc", "opt0", "measPE"),                 "opt0 measured PE",          None,                     True),
    ("opt0_hypoPE",         c("slc", "opt0", "hypoPE"),                 "opt0 hypothesis PE",        None,                     True),
    ("crlongtrkdiry",       c("slc", "nuid", "crlongtrkdiry"),          "CR long-track dir. y",      None,                     False),
    ("vtx_x",               c("slc", "vertex", "x"),                    "Slice vertex x [cm]",       np.linspace(-200, 200, 41), False),
    ("vtx_y",               c("slc", "vertex", "y"),                    "Slice vertex y [cm]",       np.linspace(-200, 200, 41), False),
    ("vtx_z",               c("slc", "vertex", "z"),                    "Slice vertex z [cm]",       np.linspace(0, 500, 51),  False),
    ("n_trk",               c("n_trk"),                                 "Track-like PFPs per slice", np.arange(-0.5, 12.5, 1), False),
]
for name, col, xlabel, bins, logy in SLICE_VARS:
    plot("slc_" + name, col, xlabel, bins, logy)

## 5. Longest and second-longest track per slice
Ranked by length only; not a muon/pion assignment.

In [ ]:
TRACK_VARS = [
    ("len",                 ("trk", "len"),                       "Track length [cm]",             np.linspace(0, 400, 81), True),
    ("trackScore",          ("trackScore",),                      "Track score",                   np.linspace(0.5, 1, 51), True),
    ("theta_z",             ("theta_z",),                         "Angle to z [deg]",              np.linspace(0, 180, 37), False),
    ("dist_vtx",            ("dist_vtx",),                        "Start-to-slice-vertex [cm]",    np.linspace(0, 100, 51), True),
    ("rangeP_mu",           ("trk", "rangeP", "p_muon"),          "rangeP (μ hypothesis) [GeV/c]", np.linspace(0, 2, 51),   True),
    ("rangeP_pi",           ("trk", "rangeP", "p_pion"),          "rangeP (π hypothesis) [GeV/c]", np.linspace(0, 2, 51),   True),
    ("chi2_muon_pl2",       ("trk", "chi2pid", "I2", "chi2_muon"),   "χ²(μ), plane 2",   None, True),
    ("chi2_pion_pl2",       ("trk", "chi2pid", "I2", "chi2_pion"),   "χ²(π), plane 2",   None, True),
    ("chi2_proton_pl2",     ("trk", "chi2pid", "I2", "chi2_proton"), "χ²(p), plane 2",   None, True),
    ("pida_pl2",            ("trk", "chi2pid", "I2", "pida"),        "PIDA, plane 2",    None, True),
    ("chi2_mu_over_pi_pl2", ("chi2_mu_over_pi_pl2",),             "χ²(μ)/χ²(π), plane 2",          np.linspace(0, 5, 51),   True),
]
for level, tag in [("long1", "Longest"), ("long2", "2nd-longest")]:
    for name, sub, xlabel, bins, logy in TRACK_VARS:
        plot(f"{level}_{name}", c(level, *sub), f"{tag} track: {xlabel}", bins, logy)

## 6. Efficiency, purity and S/√B as the cuts are added
Cumulative over `CUTS[m]` (`DEFAULT_CUTS`, then the track cuts at that mass's thresholds, containment last), one flow per mass with its own
background, joined on the stage. First point = cafpyana preselection. The last two points are the **split**: contained and uncontained
(open diamonds forking from the Michel veto, shaded band). The x labels name the cut quantity only (`ana.CUT_SHORT_LABELS`); thresholds differ per mass.

* Efficiency = rate-weighted (`weights_mc`) HNL-matched slices after the cut / the same at the preselection. Relative to the preselection: these frames have no generated-level denominator.
* Purity = S / (S + B) at the data POT. S = HNL-matched slices at the simulated |U_μ4|² (no `HNL_DISPLAY_SCALE`); B = BNB ν + CORSIKA MC **plus off-beam** (`B bnb`, `B off` columns), all scaled to the data POT. `hnlcosmic` slices are left out (their cosmics are already in B).
  The rate goes as |U|⁴, so the absolute purity is set by the chosen coupling: only the cut-to-cut trend means something. The legend gives |U|², the plot the POT.
* S/√B relative to the preselection: independent of the coupling and the POT (both scale out of the ratio).
* Split table: S, B, raw B slices and S/√B per category, and the combination sqrt(S_c²/B_c + S_u²/B_u).
* Statistical only, no systematics. PDFs: `PLOT_DIR/cutflow_{efficiency,purity,ssb_rel}.pdf`.

In [ ]:
flows = {m: ana.eff_purity_flow({f"m{m}": hnl0[m]}, {"bnb": bnb0[m], "off": off0[m]}, CUTS[m],
                                sig_scale={f"m{m}": W_HNL[m]}, bkg_scale=W, split=True) for m in MASSES}
flow = pd.concat([flows[m].drop(columns=["cut", "branch"]).add_prefix(f"[m{m}] ") for m in MASSES], axis=1)
flow.insert(0, "branch", flows[MASSES[0]]["branch"])
for m in MASSES:     # the plot reads "<metric> <key>" columns: one key per mass
    for metric in ("S", "eff", "purity", "ssb_rel"):
        flow[f"{metric} m{m}"] = flows[m][f"{metric} m{m}"]
xlabels = [ana.CUT_SHORT_LABELS[s] for s in flow.index]   # cut quantity only, no thresholds
display(flow.style.format("{:.3g}"))
flow_labels = {f"m{m}": ana.hnl_label(m, HNL_INFO[m]["simU"]) for m in MASSES}
pot_note = f"{POT_DATA:.2e} POT (Run 1 data)"
for metric, fname, ylabel, logy, note in [
        ("eff",     "efficiency", "HNL efficiency (relative to preselection)",  False, None),
        ("purity",  "purity",     "Purity S / (S + B)",                         True,  pot_note),
        ("ssb_rel", "ssb_rel",    r"S/$\sqrt{B}$ relative to preselection",      False, None)]:
    fig, _ = ana.plot_cut_progression(flow, metric, ylabel, labels=flow_labels, log_y=logy, note=note, xlabels=xlabels,
                                      savefig=PLOT_DIR / f"cutflow_{fname}.pdf")
    plt.show()
    plt.close(fig)

split = pd.concat([ana.split_table({f"m{m}": hnl0[m]}, {"bnb": bnb0[m], "off": off0[m]}, CUTS[m],
                                   sig_scale={f"m{m}": W_HNL[m]}, bkg_scale=W) for m in MASSES])
display(split.style.format("{:.3g}"))